In [9]:
# # CIFAR-10 CNN in PyTorch
# This notebook trains a CNN on the CIFAR-10 dataset using PyTorch.

In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import pandas as pd
import os

In [11]:
# Hyperparameters
batch_size = 256
num_classes = 10
epochs = 100
learning_rate = 0.001
weight_decay = 1e-4
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [12]:
# Data Loading
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

train_dataset = datasets.CIFAR10(root='data', train=True, download=True, transform=transform)
test_dataset  = datasets.CIFAR10(root='data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)


100.0%
c:\Users\knith\AppData\Local\Programs\Python\Python314\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [13]:
# Model Definition

class CIFAR10_CNN(nn.Module):
    def __init__(self, num_classes=10, weight_decay=1e-4):
        super(CIFAR10_CNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.conv5 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.conv6 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2,2)
        self.dropout = nn.Dropout(0.6)
        self.fc1 = nn.Linear(128*4*4, 512)
        self.fc2 = nn.Linear(512, num_classes)
        self.elu = nn.ELU()

    def forward(self, x):
        x = self.elu(self.conv1(x))
        x = self.elu(self.conv2(x))
        x = self.pool(x)
        x = self.dropout(x)

        x = self.elu(self.conv3(x))
        x = self.elu(self.conv4(x))
        x = self.pool(x)
        x = self.dropout(x)

        x = self.elu(self.conv5(x))
        x = self.elu(self.conv6(x))
        x = self.pool(x)
        x = self.dropout(x)

        x = x.view(x.size(0), -1)
        x = self.elu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

In [14]:
model = CIFAR10_CNN(num_classes=num_classes, weight_decay=weight_decay).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

In [15]:
# Training Loop

train_losses = []
train_accuracies = []

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for i, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    train_losses.append(epoch_loss)
    train_accuracies.append(epoch_acc)
    
    if (epoch+1) % 10 == 0 or epoch==0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.4f}")

Epoch [1/100], Loss: 1.5931, Accuracy: 0.4185
Epoch [10/100], Loss: 0.9122, Accuracy: 0.6798
Epoch [20/100], Loss: 0.8195, Accuracy: 0.7168
Epoch [30/100], Loss: 0.7701, Accuracy: 0.7351
Epoch [40/100], Loss: 0.7366, Accuracy: 0.7483
Epoch [50/100], Loss: 0.7132, Accuracy: 0.7577
Epoch [60/100], Loss: 0.6888, Accuracy: 0.7661
Epoch [70/100], Loss: 0.6828, Accuracy: 0.7702
Epoch [80/100], Loss: 0.6557, Accuracy: 0.7781
Epoch [90/100], Loss: 0.6517, Accuracy: 0.7813
Epoch [100/100], Loss: 0.6474, Accuracy: 0.7825


In [16]:
# Evaluation on Test Data

model.eval()
correct = 0
total = 0
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

test_accuracy = correct / total
print("Test Accuracy:", test_accuracy)

Test Accuracy: 0.8296


In [17]:
# Save Results to Excel

output_file = 'data/Output_PyTorch_CIFAR10.xlsx'
os.makedirs('data', exist_ok=True)

df_results = pd.DataFrame({
    'Epoch': list(range(1, epochs+1)),
    'Train_Loss': train_losses,
    'Train_Accuracy': train_accuracies
})

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    df_results.to_excel(writer, sheet_name='Train_Metrics', index=False)
    pd.DataFrame({'Test_Accuracy':[test_accuracy]}).to_excel(writer, sheet_name='Test_Metrics', index=False)

print("Results saved to", output_file)

Results saved to data/Output_PyTorch_CIFAR10.xlsx
